# Phase 6 — Model Development

This notebook contains the reproducible machine-learning workflow for Student Learning State Identification.


In [1]:
import pandas as pd
from pathlib import Path
print('='*90+'\nPHASE 6 â€” STEP 1: MODELLING DATASET PREPARATION & VALIDATION\n'+'='*90)
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
path = project_root / 'data' / 'processed' / 'learning_state_labelled.csv';d=pd.read_csv(path,low_memory=False);predictors=['previous_interaction_count','previous_skill_interaction_count','recent_accuracy_10','recent_accuracy_change_5_vs_10','recent_attempt_log_mean_10','recent_multi_attempt_rate_10','recent_hint_usage_rate_10','recent_hint_available_rate_10','recent_response_log_mean_10','recent_response_median_ms_10','recent_skill_accuracy_5','recent_skill_accuracy_change_3_vs_previous_2','has_overall_history','has_skill_history','has_full_skill_window_5','has_recent_hint_usage_evidence'];target='learning_state'
print('\nSOURCE DATASET\n'+'='*90);print('Path:',path);print('Shape:',d.shape);print('Rows:',len(d));print('Columns:',len(d.columns));print('\nAPPROVED PREDICTORS\n'+'='*90)
for i,x in enumerate(predictors,1):print(f'{i:2d}. {x}')
print('\nNumber of predictors:',len(predictors));missing=[x for x in predictors if x not in d];print('\nPREDICTOR EXISTENCE CHECK\n'+'='*90);print('Missing approved predictors:',missing);print('All 16 predictors exist:',not missing)
states=['NEEDS_SUPPORT','DEVELOPING','STRONG'];e=d[d[target].isin(states)].copy();c=d[d[target]=='UNAVAILABLE'].copy();print('\nROW SEPARATION\n'+'='*90);print('Total rows:',len(d));print('Eligible modelling rows:',len(e));print('Cold-start rows:',len(c));print('Eligible + cold-start = total:',len(e)+len(c)==len(d));print('\nELIGIBLE TARGET DISTRIBUTION\n'+'='*90);z=e[target].value_counts();print(pd.DataFrame({'interactions':z,'percentage':(e[target].value_counts(normalize=True)*100).round(2)}));print('\nOnly 3 modelling states present:',set(e[target].unique())==set(states));print('UNAVAILABLE excluded from modelling rows:','UNAVAILABLE' not in e[target].unique())
print('\nCOLD-START VALIDATION\n'+'='*90);print('All cold-start rows labelled UNAVAILABLE:',c[target].eq('UNAVAILABLE').all());print('All cold-start rows have no overall history:',c.has_overall_history.eq(0).all());print('All cold-start rows have no skill history:',c.has_skill_history.eq(0).all());print('All cold-start rows have previous_interaction_count = 0:',c.previous_interaction_count.eq(0).all());print('All cold-start rows have previous_skill_interaction_count = 0:',c.previous_skill_interaction_count.eq(0).all())
X=e[predictors].copy();y=e[target].copy();print('\nMODELLING MATRICES\n'+'='*90);print('X shape:',X.shape);print('y shape:',y.shape);print('X rows equal y rows:',len(X)==len(y));print('X contains exactly 16 columns:',X.shape[1]==16)
forbid=['correct','attempt_count','hint_count','hint_total','ms_first_response','overlap_time','bottom_hint','first_action'];ids=['order_id','user_id','assignment_id','problem_id','skill_id','skill_name'];ctx=['primary_performance','performance_source','evidence_level','evidence_strength','trend_context','support_context'];print('\nLEAKAGE / EXCLUSION CHECKS\n'+'='*90);print('Current correct excluded:','correct' not in X);print('No direct current behavioural fields:',not any(x in X for x in forbid));print('No identifiers used as predictors:',not any(x in X for x in ids));print('No derived label-context columns used as predictors:',not any(x in X for x in ctx));print('Target not included in predictors:',target not in X)
print('\nPREDICTOR MISSINGNESS â€” ELIGIBLE ROWS\n'+'='*90);print(pd.DataFrame({'missing':X.isna().sum(),'percentage':(X.isna().mean()*100).round(2)}).sort_values(['missing','percentage'],ascending=False));print('\nEVIDENCE LEVELS IN ELIGIBLE MODELLING DATA\n'+'='*90);print(e.evidence_level.value_counts());print('\nPercentage:');print((e.evidence_level.value_counts(normalize=True)*100).round(2));print('\nSTUDENT COVERAGE\n'+'='*90);print('Unique students in eligible modelling data:',e.user_id.nunique());print('Unique students in cold-start data:',c.user_id.nunique());print('\nEligible interactions per student:');print(e.groupby('user_id').size().describe().round(2))
checks={'Source rows = 338001':len(d)==338001,'Source columns = 30':d.shape[1]==30,'Eligible rows = 333061':len(e)==333061,'Cold-start rows = 4940':len(c)==4940,'Eligible + cold-start = source':len(e)+len(c)==len(d),'All 16 approved predictors exist':not missing,'X contains 16 predictors':X.shape[1]==16,'X and y row counts match':len(X)==len(y),'Only 3 target states in modelling data':set(y.unique())==set(states),'UNAVAILABLE excluded from modelling data':'UNAVAILABLE' not in y.unique(),'Current correct excluded':'correct' not in X,'No direct current behavioural leakage':not any(x in X for x in forbid),'No identifier predictors':not any(x in X for x in ids),'No derived label-context predictors':not any(x in X for x in ctx),'Target excluded from X':target not in X,'Cold-start has no overall history':c.has_overall_history.eq(0).all(),'Cold-start has no skill history':c.has_skill_history.eq(0).all()};v=pd.DataFrame(checks.items(),columns=['validation','passed']);print('\nFINAL STEP 1 VALIDATION\n'+'='*90);print(v.to_string(index=False));print(f'\nChecks passed: {int(v.passed.sum())} / {len(v)}');print('\nPHASE 6 â€” STEP 1 RESULT:','PASS' if v.passed.all() else 'REVIEW REQUIRED');print('\n'+'='*90+'\nPHASE 6 â€” STEP 1 COMPLETE\n'+'='*90)






PHASE 6 — STEP 1: MODELLING DATASET PREPARATION & VALIDATION

SOURCE DATASET
Path: D:\TEA_CUPS\RESEARCH_01\student_personalization_memory\data\processed\learning_state_labelled.csv
Shape: (338001, 30)
Rows: 338001
Columns: 30

APPROVED PREDICTORS
 1. previous_interaction_count
 2. previous_skill_interaction_count
 3. recent_accuracy_10
 4. recent_accuracy_change_5_vs_10
 5. recent_attempt_log_mean_10
 6. recent_multi_attempt_rate_10
 7. recent_hint_usage_rate_10
 8. recent_hint_available_rate_10
 9. recent_response_log_mean_10
10. recent_response_median_ms_10
11. recent_skill_accuracy_5
12. recent_skill_accuracy_change_3_vs_previous_2
13. has_overall_history
14. has_skill_history
15. has_full_skill_window_5
16. has_recent_hint_usage_evidence

Number of predictors: 16

PREDICTOR EXISTENCE CHECK
Missing approved predictors: []
All 16 predictors exist: True

ROW SEPARATION
Total rows: 338001
Eligible modelling rows: 333061
Cold-start rows: 4940
Eligible + cold-start = total: True

ELIGIBL

In [2]:
import pandas as pd
from pathlib import Path
root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd();d=pd.read_csv(root/'data/processed/learning_state_labelled.csv',low_memory=False);target='learning_state';eligible_states=['NEEDS_SUPPORT','DEVELOPING','STRONG'];eligible_df=d[d[target].isin(eligible_states)].copy()
print('='*90+'\nPHASE 6 â€” STEP 2: STUDENT-AWARE SPLIT INVESTIGATION\n'+'='*90);print('\nELIGIBLE MODELLING STRUCTURE\n'+'='*90);print('Eligible rows:',len(eligible_df));print('Unique students:',eligible_df.user_id.nunique());print('Target states:',sorted(eligible_df[target].unique()))
s=eligible_df.groupby('user_id').size().rename('interaction_count');print('\nELIGIBLE INTERACTIONS PER STUDENT\n'+'='*90);print(s.describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]).round(2));print('\nStudents by interaction-count group:');print(pd.cut(s,bins=[0,5,10,25,50,100,float('inf')],labels=['1-5','6-10','11-25','26-50','51-100','101+'],include_lowest=True).value_counts().sort_index())
ct=pd.crosstab(eligible_df.user_id,eligible_df[target]);
for x in eligible_states:
 if x not in ct:ct[x]=0
ct=ct[eligible_states];presence=ct.gt(0);ct['number_of_states_present']=presence.sum(axis=1);print('\nNUMBER OF LEARNING STATES REPRESENTED PER STUDENT\n'+'='*90);print(ct.number_of_states_present.value_counts().sort_index());print('\nPercentage:');print((ct.number_of_states_present.value_counts(normalize=True).sort_index()*100).round(2));print('\nSTUDENTS REPRESENTING EACH STATE\n'+'='*90)
for x in eligible_states:
 n=int(presence[x].sum());print(f'{x}: {n} students ({n/len(presence)*100:.2f}%)')
comb=ct[eligible_states].apply(lambda r:' + '.join(x for x in eligible_states if r[x]>0),axis=1);cc=comb.value_counts();print('\nSTATE COMBINATIONS PER STUDENT\n'+'='*90);print(pd.DataFrame({'students':cc,'percentage':(cc/len(comb)*100).round(2)}))
only=ct[eligible_states].copy();ct['dominant_state']=only.idxmax(axis=1);ct['dominant_state_count']=only.max(axis=1);ct['total_interactions']=only.sum(axis=1);ct['dominant_state_fraction']=ct.dominant_state_count/ct.total_interactions;dc=ct.dominant_state.value_counts();print('\nDOMINANT STATE PER STUDENT\n'+'='*90);print(pd.DataFrame({'students':dc,'percentage':(ct.dominant_state.value_counts(normalize=True)*100).round(2)}));print('\nDominant-state fraction summary:');print(ct.dominant_state_fraction.describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]).round(3))
single=ct[ct.number_of_states_present==1];print('\nSINGLE-STATE STUDENTS\n'+'='*90);print('Number:',len(single));print('\nDominant state among single-state students:');print(single.dominant_state.value_counts());print('\nTotal eligible interactions contributed:');print(single.total_interactions.describe().round(2))
largest=ct.sort_values('total_interactions',ascending=False).head(20);print('\nTOP 20 STUDENTS BY ELIGIBLE INTERACTION COUNT\n'+'='*90);print(largest[eligible_states+['total_interactions','dominant_state','dominant_state_fraction']].to_string());n=int(largest.total_interactions.sum());print('\nInteractions contributed by top 20 students:',n);print('Percentage of eligible dataset:',round(n/len(eligible_df)*100,2),'%')
checks={f'{x} represented by >=100 students':int(presence[x].sum())>=100 for x in eligible_states};checks['At least 1000 eligible students']=eligible_df.user_id.nunique()>=1000;checks['All 3 states exist in eligible data']=set(eligible_df[target].unique())==set(eligible_states);v=pd.DataFrame(checks.items(),columns=['validation','passed']);print('\nSTUDENT-LEVEL SPLIT FEASIBILITY CHECKS\n'+'='*90);print(v.to_string(index=False));print('\nChecks passed:',int(v.passed.sum()),'/',len(v));print('\nMETHODological INTERPRETATION\n'+'='*90);print('This step does NOT create train/validation/test sets.');print('It evaluates whether splitting by user_id is feasible while retaining broad representation of all three learning states.');print('The same user_id must not later appear across multiple model-evaluation partitions.');print('\n'+'='*90+'\nPHASE 6 â€” STEP 2 INVESTIGATION COMPLETE\n'+'='*90)



PHASE 6 — STEP 2: STUDENT-AWARE SPLIT INVESTIGATION

ELIGIBLE MODELLING STRUCTURE
Eligible rows: 333061
Unique students: 4030
Target states: ['DEVELOPING', 'NEEDS_SUPPORT', 'STRONG']

ELIGIBLE INTERACTIONS PER STUDENT
count    4030.00
mean       82.65
std       164.17
min         1.00
10%         4.00
25%         9.00
50%        23.00
75%        72.00
90%       197.00
95%       434.95
99%       851.42
max      1294.00
Name: interaction_count, dtype: float64

Students by interaction-count group:
interaction_count
1-5       622
6-10      545
11-25     938
26-50     680
51-100    452
101+      793
Name: count, dtype: int64

NUMBER OF LEARNING STATES REPRESENTED PER STUDENT
number_of_states_present
1     671
2     660
3    2699
Name: count, dtype: int64

Percentage:
number_of_states_present
1    16.65
2    16.38
3    66.97
Name: proportion, dtype: float64

STUDENTS REPRESENTING EACH STATE
NEEDS_SUPPORT: 3347 students (83.05%)
DEVELOPING: 3267 students (81.07%)
STRONG: 3474 students (86.20%